## Dataset Unification and Preprocessing
This notebook aims to unify three datasets: TCGA-LUAD, TCGA-LUSC, MSK-CHORD-2024. These datasets are then preprocessed as required and split into training and testing sets for further processing.


TCGA-LUAD and TCGA-LUSC are TCGA's two major non-small cell lung cancer cohorts — lung adenocarcinoma (585 patients) and lung squamous cell carcinoma (504 patients), respectively — each providing multi-omic and clinical characterization (clinical and biospecimen annotations, pathology reports, treatment/drug and radiation therapy records, and molecular profiling data) collected through the TCGA program to enable histology-specific study of NSCLC subtypes; MSK-CHORD-2024, by contrast, is a real-world clinicogenomic dataset from Memorial Sloan Kettering Cancer Center covering 24,950 patients across five cancer types, including 7,809 with non-small-cell lung cancer, combining natural language processing–derived clinical annotations with structured medication, demographic, tumor registry, and tumor genomic data from targeted MSK-IMPACT sequencing to support large-scale genotype–phenotype and outcome-prediction research.

### 1. Load the per-source cleaned outputs
`explore_tcga.ipynb` produced `datasets/tcga_nsclc_clinical_clean.csv` (one row per TCGA-LUAD/LUSC patient) and `explore_mskchord.ipynb` produced `datasets/msk_chord_2024/msk_chord_out/nsclc_patient_level_features.csv` (one row per MSK-CHORD NSCLC patient, already joined with sample-level and timeline-derived features). Both are loaded here as-is before any column harmonization.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

TCGA_CLEAN = "datasets/tcga_nsclc_clinical_clean.csv"
MSK_FEATURES = "datasets/msk_chord_2024/msk_chord_out/nsclc_patient_level_features.csv"
OUT_DIR = Path("datasets/unified")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
tcga = pd.read_csv(TCGA_CLEAN)
msk = pd.read_csv(MSK_FEATURES)

print("TCGA:", tcga.shape)
print(tcga.columns.tolist())
print()
print("MSK-CHORD:", msk.shape)
print(msk.columns.tolist())

TCGA: (1026, 27)
['bcr_patient_barcode', 'HISTOLOGY', 'gender', 'race', 'ethnicity', 'age_at_initial_pathologic_diagnosis', 'pathologic_stage', 'pathologic_T', 'pathologic_N', 'pathologic_M', 'histological_type', 'icd_o_3_histology', 'tumor_tissue_site', 'tobacco_smoking_history', 'number_pack_years_smoked', 'egfr_mutation_result', 'kras_mutation_result', 'eml4_alk_translocation_performed', 'vital_status', 'days_to_death', 'days_to_last_followup', 'eastern_cancer_oncology_group', 'karnofsky_performance_score', 'radiation_therapy', 'new_tumor_event_after_initial_treatment', 'days_to_event', 'OS_STATUS']

MSK-CHORD: (7809, 53)
['PATIENT_ID', 'GENDER', 'RACE', 'ETHNICITY', 'CURRENT_AGE_DEID', 'STAGE_HIGHEST_RECORDED', 'NUM_ICDO_DX', 'ADRENAL_GLANDS', 'BONE', 'CNS_BRAIN', 'INTRA_ABDOMINAL', 'LIVER', 'LUNG', 'LYMPH_NODES', 'OTHER', 'PLEURA', 'REPRODUCTIVE_ORGANS', 'SMOKING_PREDICTIONS_3_CLASSES', 'GLEASON_FIRST_REPORTED', 'GLEASON_HIGHEST_REPORTED', 'HISTORY_OF_PDL1', 'PRIOR_MED_TO_MSK', 'O

### 2. Resolve column names on the MSK-CHORD side
The sample-level columns were confirmed during exploration (`CANCER_TYPE`, `TUMOR_PURITY`, `TMB_NONSYNONYMOUS`, etc.), but the patient-level clinical columns (age, gender, race, smoking, survival) were never printed in `explore_mskchord.ipynb` — only `data_clinical_sample.txt` was inspected there, not `data_clinical_patient.txt`'s full header. Rather than assume exact field names (they vary across cBioPortal study releases), resolve them defensively against a list of plausible candidates and flag anything that doesn't match, following the same "check the exact column name/values first" pattern used in `filter.py` and both exploration notebooks.

In [ ]:
def pick_col(df, candidates):
    """Return the first column name from `candidates` that exists in df, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

# Candidate cBioPortal clinical_patient header names for each unified field.
# Update this against the printed msk.columns.tolist() above if any come back None.
#
# NOTE: subtype_target / stage_target / pdl1_status_category were assumed (per
# explore_mskchord.ipynb's docstring) to already exist as engineered columns on
# nsclc_patient_level_features.csv, but they are NOT in the printed column list
# above -- only the raw cBioPortal fields they'd have been derived from are.
# Resolving them here defensively too, against their nearest raw equivalents,
# instead of assuming the engineered names, so this doesn't crash on load.
msk_field_candidates = {
    "AGE":              ["CURRENT_AGE_DEID", "AGE", "AGE_AT_DIAGNOSIS", "AGE_AT_DX"],
    "GENDER":           ["GENDER", "SEX"],
    "RACE":             ["RACE"],
    "ETHNICITY":        ["ETHNICITY"],
    "SMOKING_STATUS":   ["SMOKING_HISTORY", "SMOKING_STATUS", "SMOKING", "SMOKING_PREDICTIONS_3_CLASSES"],
    "PACK_YEARS":       ["SMOKING_PACK_YEARS", "PACK_YEARS"],
    "OS_STATUS":        ["OS_STATUS", "VITAL_STATUS"],
    "OS_MONTHS":        ["OS_MONTHS"],
    "HISTOLOGY":        ["subtype_target", "CANCER_TYPE_DETAILED", "ONCOTREE_CODE", "CANCER_TYPE"],
    "STAGE_GROUP":      ["stage_target", "STAGE_HIGHEST_RECORDED"],
    "PDL1_STATUS":      ["pdl1_status_category", "ever_pdl1_positive", "PDL1_POSITIVE", "HISTORY_OF_PDL1"],
}

msk_resolved = {k: pick_col(msk, v) for k, v in msk_field_candidates.items()}
print(msk_resolved)

unresolved = [k for k, v in msk_resolved.items() if v is None]
if unresolved:
    print("No match found for:", unresolved, "— inspect msk.columns.tolist() above and extend the candidates")


### 3. Build the TCGA half of the unified table
TCGA reports fine-grained AJCC stage strings (`Stage IA`, `Stage IIIB`, ...). MSK-CHORD's `stage_target` (built in `explore_mskchord.ipynb`) already collapses to the coarser `Stage 1-3` / `Stage 4` / `Unknown` buckets, so TCGA's stage is coarsened the same way here to keep the two sources comparable.

In [5]:
def coarsen_tcga_stage(stage):
    if not isinstance(stage, str):
        return "Unknown"
    s = stage.upper()
    if "IV" in s:
        return "Stage 4"
    if "I" in s:   # covers I, IA, IB, II, IIA, IIB, III, IIIA, IIIB (IV already handled above)
        return "Stage 1-3"
    return "Unknown"

tcga_unified = pd.DataFrame({
    "PATIENT_ID": tcga["bcr_patient_barcode"],
    "SOURCE": "TCGA",
    "HISTOLOGY": tcga["HISTOLOGY"],                 # already LUAD / LUSC
    "GENDER": tcga["gender"],
    "RACE": tcga["race"],
    "ETHNICITY": tcga["ethnicity"],
    "AGE": tcga["age_at_initial_pathologic_diagnosis"],
    "SMOKING_STATUS": tcga["tobacco_smoking_history"],
    "PACK_YEARS": tcga["number_pack_years_smoked"],
    "STAGE_GROUP": tcga["pathologic_stage"].apply(coarsen_tcga_stage),
    "OS_STATUS": tcga["OS_STATUS"],
    "OS_DAYS": tcga["days_to_event"],
    "EGFR_RESULT": tcga["egfr_mutation_result"],
    "KRAS_RESULT": tcga["kras_mutation_result"],
})

print(tcga_unified.shape)
print(tcga_unified["STAGE_GROUP"].value_counts(dropna=False))

(1026, 14)
STAGE_GROUP
Stage 1-3    981
Stage 4       33
Unknown       12
Name: count, dtype: int64


### 4. Build the MSK-CHORD half of the unified table
`subtype_target`, `stage_target`, `pdl1_status_category`, `num_treatment_events`, and `num_distinct_tumor_sites` were already engineered in `explore_mskchord.ipynb`, so they're reused directly rather than recomputed.

**Deliberately excluded: `latest_ecog_status`.** Exploration found `ECOG` is null for every row of `data_timeline_performance_status.txt` — 0 non-null out of 157,674 rows in the *full, unfiltered* file, not just the NSCLC subset — so it carries zero signal in this release of MSK-CHORD and would only look like real, informative missingness downstream. It's left out of the unified table rather than silently imputed.

In [ ]:
msk_unified = pd.DataFrame({
    "PATIENT_ID": msk["PATIENT_ID"],
    "SOURCE": "MSK_CHORD",
    "NUM_TREATMENT_EVENTS": msk["num_treatment_events"],
    "NUM_DISTINCT_TUMOR_SITES": msk["num_distinct_tumor_sites"],
    "TUMOR_PURITY": msk["TUMOR_PURITY"],
    "TMB": msk["TMB_NONSYNONYMOUS"],
    "MSI_SCORE": msk["MSI_SCORE"],
})

# HISTOLOGY, STAGE_GROUP, and PDL1_STATUS are filled from msk_resolved below along
# with the rest, since subtype_target / stage_target / pdl1_status_category turned
# out not to exist as such on this CSV -- see the note in step 2. If any of these
# three resolved to a raw cBioPortal column instead of the expected engineered one,
# the values will need the same derivation logic that explore_mskchord.ipynb applied
# (e.g. collapsing ONCOTREE_CODE/CANCER_TYPE_DETAILED into the LUAD/LUSC/Other
# buckets, or STAGE_HIGHEST_RECORDED into Stage 1-3/Stage 4/Unknown) before they're
# truly comparable to the TCGA side -- check msk_resolved's printed output above.
for unified_name, source_col in msk_resolved.items():
    msk_unified[unified_name] = msk[source_col] if source_col is not None else np.nan

print(msk_unified.shape)
print(msk_unified.isna().mean().sort_values(ascending=False))


### 5. Align histology labels across sources
TCGA only distinguishes LUAD vs LUSC (that's the cohort split). MSK-CHORD's `subtype_target` has five buckets (`Lung Adenocarcinoma`, `Lung Squamous Cell Carcinoma`, `Non-Small Cell Lung Cancer`, `Large Cell Neuroendocrine Carcinoma`, `Other`). Map the matching two onto TCGA's labels and bucket everything else as `Other_NSCLC` so the combined `HISTOLOGY` column means the same thing regardless of source.

In [ ]:
histology_map = {
    "Lung Adenocarcinoma": "LUAD",
    "Lung Squamous Cell Carcinoma": "LUSC",
}
msk_unified["HISTOLOGY"] = msk_unified["HISTOLOGY"].map(histology_map).fillna("Other_NSCLC")

print("MSK-CHORD histology after mapping:")
print(msk_unified["HISTOLOGY"].value_counts())
print("\nTCGA histology:")
print(tcga_unified["HISTOLOGY"].value_counts())

### 6. Concatenate and save the unified table

In [ ]:
common_cols = sorted(set(tcga_unified.columns) & set(msk_unified.columns))
print("Shared columns:", common_cols)
print("TCGA-only columns:", sorted(set(tcga_unified.columns) - set(msk_unified.columns)))
print("MSK-only columns:", sorted(set(msk_unified.columns) - set(tcga_unified.columns)))

unified = pd.concat([tcga_unified, msk_unified], axis=0, ignore_index=True, sort=False)
print(unified.shape)
print(unified["SOURCE"].value_counts())

unified.to_csv(OUT_DIR / "unified_nsclc_clinical_raw.csv", index=False)

### 7. Harmonize smoking-status vocabulary
TCGA's `tobacco_smoking_history` uses the TCGA CDE numeric codes (1 = Lifelong non-smoker, 2 = Current smoker, 3-5 = various former-smoker categories). MSK-CHORD's smoking field is expected to already be plain-text (`Never` / `Current` / `Former`). **Print both value sets before trusting the mapping below** — if MSK-CHORD's resolved column turned out to be `None` in step 2, or uses different text, this needs adjusting first.

In [ ]:
print("TCGA smoking values:", sorted(unified.loc[unified.SOURCE == "TCGA", "SMOKING_STATUS"].dropna().unique()))
print("MSK-CHORD smoking values:", sorted(unified.loc[unified.SOURCE == "MSK_CHORD", "SMOKING_STATUS"].dropna().unique()))

tcga_smoking_map = {
    1: "Never",
    2: "Current",
    3: "Former",
    4: "Former",
    5: "Former",
}
is_tcga = unified["SOURCE"] == "TCGA"
unified.loc[is_tcga, "SMOKING_STATUS"] = unified.loc[is_tcga, "SMOKING_STATUS"].map(tcga_smoking_map)

print()
print(unified["SMOKING_STATUS"].value_counts(dropna=False))

### 8. Stratified train / val / test split
Rows with `STAGE_GROUP == "Unknown"` are dropped for modeling, mirroring the rule already applied to MSK-CHORD alone in `explore_mskchord.ipynb`. Stratifying on `SOURCE` + `STAGE_GROUP` jointly keeps both the TCGA/MSK-CHORD mix and the stage balance consistent across splits, since the two cohorts differ a lot in size (1,026 vs 7,809 patients) and TCGA has no `Unknown` stage rows at all.

In [ ]:
from sklearn.model_selection import train_test_split

model_df = unified[unified["STAGE_GROUP"] != "Unknown"].copy()
strat_key = model_df["SOURCE"] + "_" + model_df["STAGE_GROUP"]

train_df, temp_df = train_test_split(
    model_df, test_size=0.30, stratify=strat_key, random_state=42
)
temp_strat_key = temp_df["SOURCE"] + "_" + temp_df["STAGE_GROUP"]
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_strat_key, random_state=42
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print()
print(train_df.groupby(["SOURCE", "STAGE_GROUP"]).size())

train_df.to_csv(OUT_DIR / "unified_train.csv", index=False)
val_df.to_csv(OUT_DIR / "unified_val.csv", index=False)
test_df.to_csv(OUT_DIR / "unified_test.csv", index=False)

print("\nSaved to", OUT_DIR.resolve())